In [1]:
import re
import zlib
import base64

# Configuration for the Bloom Filter
# 300,000 bits = ~37.5 KB uncompressed. Very safe for HackerRank!
M_BITS = 2500000
K_HASHES = 7

def get_hash(word, i, m):
    # zlib.crc32 is deterministic across all computers and Python runs
    key = f"{word}_{i}".encode('utf-8')
    return zlib.crc32(key) % m

# 1. Read your training corpus
with open('cleaned_corpus.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read().lower()

# 2. Extract all words
words = set(re.findall(r'[a-z]+', raw_text))

# 3. Create the byte array (8 bits per byte)
byte_array = bytearray((M_BITS + 7) // 8)

# 4. Populate the Bloom Filter
for word in words:
    for i in range(1, K_HASHES + 1):
        bit_index = get_hash(word, i, M_BITS)
        byte_index = bit_index // 8
        bit_offset = bit_index % 8
        # Set the specific bit to 1
        byte_array[byte_index] |= (1 << bit_offset)

# 5. Compress and encode
compressed_bytes = zlib.compress(byte_array, level=9)
embedded_string = base64.b64encode(compressed_bytes).decode('utf-8')

print("--- COPY THE STRING BELOW INTO HACKERRANK ---")
print(f"EMBEDDED_BLOOM = '{embedded_string}'")
print(f"Size: {len(embedded_string) / 1024:.2f} KB")

--- COPY THE STRING BELOW INTO HACKERRANK ---
EMBEDDED_BLOOM = '...'
Size: 205.13 KB


In [3]:
import sys
import re
import string
import zlib
import base64

# --- PASTE YOUR GENERATED STRING HERE ---
EMBEDDED_BLOOM = embedded_string

M_BITS = 300000 
K_HASHES = 3

input_path = '/home/david/Documents/HackerRank-challenges/the-missing-characters/the-missing-characters-testcases/input/input00.txt'
with open(input_path, 'r', encoding='utf-8') as f:
    input_text = f.read().strip()

def get_hash(word, i, m):
    key = f"{word}_{i}".encode('utf-8')
    return zlib.crc32(key) % m

def is_in_bloom_filter(word, byte_array):
    """Returns True if the word is *probably* in the filter."""
    for i in range(1, K_HASHES + 1):
        bit_index = get_hash(word, i, M_BITS)
        byte_index = bit_index // 8
        bit_offset = bit_index % 8
        # If any bit is 0, the word is DEFINITELY NOT in the dictionary
        if not (byte_array[byte_index] & (1 << bit_offset)):
            return False
    return True

def solve():
    # input_text = sys.stdin.read().strip()
    if not input_text:
        return
        
    # 1. Decompress the Bloom Filter
    try:
        decompressed_bytes = base64.b64decode(EMBEDDED_BLOOM)
        bloom_array = bytearray(zlib.decompress(decompressed_bytes))
    except Exception:
        bloom_array = bytearray((M_BITS + 7) // 8)

    # 2. Clean the input text, preserving the '#'
    cleaned_input = re.sub(r'[^a-z\s#]', '', input_text.lower())
    cleaned_input = re.sub(r'\s+', ' ', cleaned_input)
    
    # Standard English letter frequencies for tie-breaking
    letter_freq = {
        'e': 13.0, 't': 9.1, 'a': 8.2, 'o': 7.5, 'i': 7.0, 'n': 6.7, 's': 6.3, 'h': 6.1, 'r': 6.0, 
        'd': 4.3, 'l': 4.0, 'c': 2.8, 'u': 2.8, 'm': 2.5, 'w': 2.4, 'f': 2.2, 'g': 2.0, 'y': 2.0, 
        'p': 1.9, 'b': 1.5, 'v': 0.98, 'k': 0.77, 'j': 0.15, 'x': 0.15, 'q': 0.09, 'z': 0.07
    }

    # 3. Process each missing character
    hashes = [i for i, char in enumerate(cleaned_input) if char == '#']

    for h_idx in hashes:
        start = h_idx
        while start > 0 and cleaned_input[start - 1] != ' ':
            start -= 1
        end = h_idx
        while end < len(cleaned_input) - 1 and cleaned_input[end + 1] != ' ':
            end += 1

        word_pattern = cleaned_input[start:end + 1]
        local_idx = h_idx - start

        # Generate candidates using the Bloom filter!
        valid_candidates = []
        if '#' not in (word_pattern[:local_idx] + word_pattern[local_idx + 1:]):
            for c in string.ascii_lowercase:
                candidate_word = word_pattern[:local_idx] + c + word_pattern[local_idx + 1:]
                if is_in_bloom_filter(candidate_word, bloom_array):
                    valid_candidates.append(c)

        # If no dictionary matches, fallback to all letters
        candidates_to_check = valid_candidates if valid_candidates else list(string.ascii_lowercase)

        best_char = None

        # If the Bloom filter isolated exactly one real word, we found our missing letter!
        if len(valid_candidates) == 1:
            best_char = valid_candidates[0]
            
        # If there are multiple possibilities, use local context counting to break the tie
        else:
            left_ctx = cleaned_input[:h_idx]
            right_ctx = cleaned_input[h_idx + 1:]
            max_len = max(len(left_ctx), len(right_ctx))
            found_contextual_match = False
            
            for w in range(min(40, max_len), 0, -1):
                lp = left_ctx[-w:] if w > 0 else ""
                rp = right_ctx[:w] if w > 0 else ""
                
                scores = {}
                for c in candidates_to_check:
                    query = lp + c + rp
                    score = cleaned_input.count(query)
                    if score > 0:
                        scores[c] = score
                
                if scores:
                    # Pick the candidate with the highest context match, tie-break with letter frequency
                    best_char = max(scores, key=lambda x: (scores[x], letter_freq.get(x, 0)))
                    found_contextual_match = True
                    break
            
            # Absolute fallback: just guess the most common English letter
            if not found_contextual_match:
                best_char = max(candidates_to_check, key=lambda x: letter_freq.get(x, 0))

        print(best_char)

if __name__ == '__main__':
    solve()

r
h
n
l
a
d
i
i
n
d


In [17]:
import numpy as np

m = 5.2 * 1e+5 # number of bits
n = 2.6 * 1e+5

k = (m/n) * np.log(2)

print(k)

1.3862943611198906


In [12]:
import lzma

with open("cleaned_corpus.txt", "rb") as f:
    data = f.read().strip()

compressed = lzma.compress(data, preset=9)

print(len(data))
print(len(compressed))

3360159
982064
